<a href="https://colab.research.google.com/github/Nayab-Gauhar/Using-GAN-to-generate-the-FM-matrix-Autism-/blob/main/autism_gan_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autism GAN Pipeline — Complete 3-Step Notebook

**Step 1:** Download ABIDE-I data & build FC matrices
**Step 2:** Train conditional WGAN-GP to synthesize FC matrices
**Step 3:** Classify ASD vs Control using real + synthetic data

## Step 0 — Install Dependencies

In [ ]:
import subprocess, sys
for pkg in ["nilearn", "numpy", "torch", "scikit-learn", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

## Step 1 — Build FC Matrix Dataset from ABIDE-I

Downloads preprocessed ABIDE-I ROI time series via `nilearn` and computes
Pearson correlation functional connectivity (FC) matrices.

In [ ]:
import os
import numpy as np
from nilearn import datasets
from nilearn.connectome import ConnectivityMeasure

# ---- config ----
ATLAS = "cc200"          # 'aal', 'ho', 'ez', 'tt', 'dosenbach160', 'cc200'
N_SUBJECTS = None        # None = all available subjects
OUT_DIR = "./fc_data"
DATA_DIR = "./abide_raw"

os.makedirs(OUT_DIR, exist_ok=True)

### Fetch ROI time series (CPAC pipeline, no bandpass, no GSR — matches paper)

In [ ]:

from nilearn import datasets
from tqdm.auto import tqdm

def _patch_target():
    """nilearn renamed its internal single-file downloader across versions."""
    try:
        from nilearn.datasets import _utils as mod
        if hasattr(mod, "fetch_single_file"):
            return mod, "fetch_single_file"       # nilearn >= ~0.10.3
    except ImportError:
        pass
    try:
        from nilearn.datasets import utils as mod
        if hasattr(mod, "_fetch_file"):
            return mod, "_fetch_file"              # older nilearn
    except ImportError:
        pass
    return None, None

def fetch_abide_with_progress(**kwargs):
    mod, attr = _patch_target()
    if mod is None:
        print("Could not hook nilearn's downloader — using default (no bar).")
        return datasets.fetch_abide_pcp(**kwargs)

    original = getattr(mod, attr)
    n_expected = kwargs.get("n_subjects")
    pbar = tqdm(total=(n_expected + 1) if n_expected else None,
                desc="Downloading ABIDE files", unit="file")

    def wrapped(*a, **kw):
        result = original(*a, **kw)
        pbar.update(1)
        return result

    setattr(mod, attr, wrapped)
    kwargs.setdefault("verbose", 0)   # mute nilearn's own per-file prints
    try:
        return datasets.fetch_abide_pcp(**kwargs)
    finally:
        setattr(mod, attr, original)  # unpatch, always
        pbar.n = pbar.total or pbar.n
        pbar.refresh()
        pbar.close()


print(f"Fetching ABIDE-I ROI time series for atlas={ATLAS} ...")
print("(First run downloads ~800+ files from AWS — may take 30-60 min)\n")

abide = fetch_abide_with_progress(
    data_dir=DATA_DIR,
    pipeline="cpac",
    band_pass_filtering=False,
    global_signal_regression=False,
    derivatives=[f"rois_{ATLAS}"],
    n_subjects=N_SUBJECTS,
    quality_checked=True,
)

roi_key = f"rois_{ATLAS}"
time_series_list = abide[roi_key]
phenotypic = abide.phenotypic

print(f"\nFetched {len(time_series_list)} subjects")
print(f"Phenotypic columns: {list(phenotypic.columns[:10])}...")

Fetching ABIDE-I ROI time series for atlas=cc200 ...
(First run downloads ~800+ files from AWS — may take 30-60 min)




Fetched 871 subjects
Phenotypic columns: ['i', 'Unnamed: 0', 'SUB_ID', 'X', 'subject', 'SITE_ID', 'FILE_ID', 'DX_GROUP', 'DSM_IV_TR', 'AGE_AT_SCAN']...


### Compute FC matrices (Pearson correlation)

In [ ]:
conn_measure = ConnectivityMeasure(kind="correlation")

fc_matrices, labels, sub_ids = [], [], []

for i, ts in enumerate(time_series_list):
    if ts is None or len(ts) == 0:
        continue
    fc = conn_measure.fit_transform([ts])[0]
    fc_matrices.append(fc.astype(np.float32))
    # BUG FIX: use .iloc[i] instead of [i] — phenotypic index starts at 1
    labels.append(phenotypic["DX_GROUP"].iloc[i])
    sub_ids.append(phenotypic["SUB_ID"].iloc[i])

fc_matrices = np.array(fc_matrices, dtype=np.float32)
labels = np.array(labels)
sub_ids = np.array(sub_ids)

print(f"FC matrices shape: {fc_matrices.shape}")

FC matrices shape: (871, 200, 200)


### Train/Test Split (80/20) & Save FC data

Splitting BEFORE training the GAN to prevent data leakage (synthetic data
must not memorize test subjects).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    fc_matrices, labels, sub_ids, test_size=0.2, stratify=labels, random_state=42
)

# Save train data
np.save(f"{OUT_DIR}/fc_matrices_train_{ATLAS}.npy", X_train)
np.save(f"{OUT_DIR}/labels_train_{ATLAS}.npy", y_train)
np.save(f"{OUT_DIR}/sub_ids_train_{ATLAS}.npy", id_train)

# Save test data
np.save(f"{OUT_DIR}/fc_matrices_test_{ATLAS}.npy", X_test)
np.save(f"{OUT_DIR}/labels_test_{ATLAS}.npy", y_test)
np.save(f"{OUT_DIR}/sub_ids_test_{ATLAS}.npy", id_test)

print(f"Total Subjects: {fc_matrices.shape[0]} (ASD: {(labels == 1).sum()}, Control: {(labels == 2).sum()})")
print(f"Training Set:   {X_train.shape[0]} subjects (ASD: {(y_train == 1).sum()}, Control: {(y_train == 2).sum()})")
print(f"Test Set:       {X_test.shape[0]} subjects (ASD: {(y_test == 1).sum()}, Control: {(y_test == 2).sum()})")

Total Subjects: 871 (ASD: 403, Control: 468)
Training Set:   696 subjects (ASD: 322, Control: 374)
Test Set:       175 subjects (ASD: 81, Control: 94)


---
## Step 2 — Conditional WGAN-GP for Synthetic FC Matrices

Trains on the real FC matrices and generates class-conditional synthetic
ASD / Control FC matrices. Only upper-triangular correlations are modeled;
full symmetric matrices (diagonal = 1.0) are reconstructed after generation.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- config ----
LATENT_DIM = 128
BATCH_SIZE = 32
EPOCHS = 2000
N_CRITIC = 5
LAMBDA_GP = 10
LR = 1e-4
N_SYNTH_PER_CLASS = 200
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)
print("Device:", DEVICE)

Device: cpu


### Load REAL TRAINING FC matrices and flatten to upper-triangular vectors
We only train the GAN on the training split to avoid data leakage!

In [ ]:
fc = np.load(f"{OUT_DIR}/fc_matrices_train_{ATLAS}.npy")
raw_labels = np.load(f"{OUT_DIR}/labels_train_{ATLAS}.npy")
gan_labels = (raw_labels == 1).astype(np.float32)  # ASD -> 1.0, Control -> 0.0

N, R, _ = fc.shape
iu = np.triu_indices(R, k=1)
vec_dim = len(iu[0])
print(f"{N} subjects | ROI={R} | vector dim={vec_dim}")


def mat_to_vec(mats):
    return np.stack([m[iu] for m in mats])


def vec_to_mat(vecs, roi=R):
    """Reconstruct full symmetric FC matrix from upper-triangular vector."""
    mats = np.zeros((len(vecs), roi, roi), dtype=np.float32)
    for i, v in enumerate(vecs):
        m = np.zeros((roi, roi), dtype=np.float32)
        m[iu] = v
        m = m + m.T
        np.fill_diagonal(m, 1.0)
        mats[i] = m
    return mats


X = mat_to_vec(fc).astype(np.float32)

696 subjects | ROI=200 | vector dim=19900


### Dataset, Generator, Discriminator

In [ ]:
class FCDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = FCDataset(X, gan_labels)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


class Generator(nn.Module):
    def __init__(self, latent_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, out_dim), nn.Tanh(),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class Discriminator(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim + 1, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))


G = Generator(LATENT_DIM, vec_dim).to(DEVICE)
D = Discriminator(vec_dim).to(DEVICE)
g_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9))
d_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.9))

### Gradient Penalty + Training Loop

In [ ]:
def gradient_penalty(D, real, fake, y):
    eps = torch.rand(real.size(0), 1, device=DEVICE)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    d_interp = D(interp, y)
    grads = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True,
    )[0]
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


print(f"Training WGAN-GP for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    for real_x, real_y in loader:
        real_x, real_y = real_x.to(DEVICE), real_y.to(DEVICE)

        for _ in range(N_CRITIC):
            z = torch.randn(real_x.size(0), LATENT_DIM, device=DEVICE)
            fake_x = G(z, real_y).detach()
            d_real = D(real_x, real_y)
            d_fake = D(fake_x, real_y)
            gp = gradient_penalty(D, real_x, fake_x, real_y)
            d_loss = d_fake.mean() - d_real.mean() + LAMBDA_GP * gp
            d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        z = torch.randn(real_x.size(0), LATENT_DIM, device=DEVICE)
        fake_x = G(z, real_y)
        g_loss = -D(fake_x, real_y).mean()
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

    if epoch % 100 == 0:
        print(f"  epoch {epoch:4d}/{EPOCHS} | d_loss={d_loss.item():.3f} | g_loss={g_loss.item():.3f}")

torch.save(G.state_dict(), f"generator_{ATLAS}.pt")
print("Generator saved.")

Training WGAN-GP for 2000 epochs...
  epoch    0/2000 | d_loss=-32.265 | g_loss=-17.814
  epoch  100/2000 | d_loss=-5.841 | g_loss=-15.616


### Generate synthetic FC matrices

In [ ]:
def synthesize(n_per_class):
    G.eval()
    with torch.no_grad():
        z1 = torch.randn(n_per_class, LATENT_DIM, device=DEVICE)
        y1 = torch.ones(n_per_class, 1, device=DEVICE)
        z0 = torch.randn(n_per_class, LATENT_DIM, device=DEVICE)
        y0 = torch.zeros(n_per_class, 1, device=DEVICE)
        vec_asd = G(z1, y1).cpu().numpy()
        vec_ctrl = G(z0, y0).cpu().numpy()
    return vec_to_mat(vec_asd), vec_to_mat(vec_ctrl)


synth_asd, synth_ctrl = synthesize(N_SYNTH_PER_CLASS)
np.save(f"{OUT_DIR}/synth_asd_{ATLAS}.npy", synth_asd)
np.save(f"{OUT_DIR}/synth_ctrl_{ATLAS}.npy", synth_ctrl)

print(f"Generated {N_SYNTH_PER_CLASS} synthetic ASD + {N_SYNTH_PER_CLASS} control FC matrices")
print(f"Symmetry check (should be 0.0): {np.abs(synth_asd[0] - synth_asd[0].T).max():.6f}")
print(f"Diagonal check (should be True): {np.allclose(np.diagonal(synth_asd[0]), 1.0)}")

### Visualize real vs synthetic FC matrices

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ["Real ASD", "Synthetic ASD", "Real Control", "Synthetic Control"]
matrices = [
    fc[raw_labels == 1][0],
    synth_asd[0],
    fc[raw_labels == 2][0],
    synth_ctrl[0],
]
for ax, mat, title in zip(axes, matrices, titles):
    im = ax.imshow(mat, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("ROI")
    ax.set_ylabel("ROI")
plt.colorbar(im, ax=axes, shrink=0.8, label="Correlation")
plt.suptitle("Functional Connectivity Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fc_comparison_{ATLAS}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved comparison plot to {OUT_DIR}/fc_comparison_{ATLAS}.png")

---
## Step 3 — Classification (ASD vs Control)

Trains multiple classifiers on:
1. **Real data only** (baseline): Trained on 80% train split.
2. **Real + GAN-synthesized data** (augmented): Trained on 80% train split + GAN samples.

Compares performance on the isolated 20% test split to show the true effect of GAN augmentation.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd
import seaborn as sns

### Prepare real and augmented datasets

In [ ]:
# Load data (use the pre-split files!)
fc_train_real = np.load(f"{OUT_DIR}/fc_matrices_train_{ATLAS}.npy")
labels_train_real = np.load(f"{OUT_DIR}/labels_train_{ATLAS}.npy")

fc_test_real = np.load(f"{OUT_DIR}/fc_matrices_test_{ATLAS}.npy")
labels_test_real = np.load(f"{OUT_DIR}/labels_test_{ATLAS}.npy")

synth_asd = np.load(f"{OUT_DIR}/synth_asd_{ATLAS}.npy")
synth_ctrl = np.load(f"{OUT_DIR}/synth_ctrl_{ATLAS}.npy")

# Flatten FC matrices to upper-triangular vectors for classifiers
iu = np.triu_indices(fc_train_real.shape[1], k=1)

X_train_real = np.stack([m[iu] for m in fc_train_real])
y_train_real = (labels_train_real == 1).astype(int)  # 1=ASD, 0=Control

X_test_real = np.stack([m[iu] for m in fc_test_real])
y_test_real = (labels_test_real == 1).astype(int)

X_synth_asd = np.stack([m[iu] for m in synth_asd])
X_synth_ctrl = np.stack([m[iu] for m in synth_ctrl])

X_train_augmented = np.vstack([X_train_real, X_synth_asd, X_synth_ctrl])
y_train_augmented = np.concatenate([
    y_train_real,
    np.ones(len(X_synth_asd), dtype=int),
    np.zeros(len(X_synth_ctrl), dtype=int),
])

print(f"Real Train dataset:      {X_train_real.shape[0]} samples ({y_train_real.sum()} ASD, {(1-y_train_real).sum()} Ctrl)")
print(f"Augmented Train dataset: {X_train_augmented.shape[0]} samples")
print(f"Real Test dataset:       {X_test_real.shape[0]} samples (never seen by GAN!)")
print(f"Feature dim:             {X_train_real.shape[1]}")

### Define classifiers

In [ ]:
classifiers = {
    "Linear SVM": SVC(kernel="linear", C=1.0, probability=True, random_state=42),
    "RBF SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "Logistic Regression (L2)": LogisticRegression(penalty="l2", C=1.0, max_iter=1000, rhttps://www.youtube.com/watch?v=dh3ilHzIRd0andom_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

### Evaluate: Real-only vs Augmented (80/20 Split)

**Important:** We evaluate strictly on the unseen 20% test split.

In [ ]:
results = {"Classifier": [], "Training Data": [], "Accuracy": [], "F1": [], "AUC": []}

for clf_name, clf_template in classifiers.items():
    for data_label, X_tr, y_tr in [
        ("Real Only", X_train_real, y_train_real),
        ("Real + Synthetic", X_train_augmented, y_train_augmented),
    ]:
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_te_scaled = scaler.transform(X_test_real)

        from sklearn.base import clone
        clf = clone(clf_template)
        clf.fit(X_tr_scaled, y_tr)

        y_pred = clf.predict(X_te_scaled)
        y_prob = clf.predict_proba(X_te_scaled)[:, 1] if hasattr(clf, "predict_proba") else y_pred.astype(float)

        acc = accuracy_score(y_test_real, y_pred)
        f1 = f1_score(y_test_real, y_pred)
        auc = roc_auc_score(y_test_real, y_prob)

        results["Classifier"].append(clf_name)
        results["Training Data"].append(data_label)
        results["Accuracy"].append(acc)
        results["F1"].append(f1)
        results["AUC"].append(auc)

        print(f"{clf_name:25s} | {data_label:17s} | Acc={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f}")

### Results Table

In [ ]:
df_results = pd.DataFrame(results)
print("\n" + "="*80)
print("CLASSIFICATION RESULTS — 80/20 Train/Test Split (No Leakage)")
print("="*80)
print(df_results.to_string(index=False, float_format="%.4f"))

### Visualization — Real vs Augmented Performance

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ["Accuracy", "F1", "AUC"]

for ax, metric in zip(axes, metrics):
    pivot = df_results.pivot(index="Classifier", columns="Training Data", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=30, color=["#4A90D9", "#E74C3C"])
    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_ylim(0.3, 1.0)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(f"ASD vs Control Classification — Real vs GAN-Augmented ({ATLAS})\nEvaluated on completely unseen 20% test split",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/classification_results_{ATLAS}_noleak.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved results plot to {OUT_DIR}/classification_results_{ATLAS}_noleak.png")

### Best Classifier Summary

In [ ]:
print("\n" + "="*60)
best_real = df_results[df_results["Training Data"] == "Real Only"].sort_values("AUC", ascending=False).iloc[0]
best_aug = df_results[df_results["Training Data"] == "Real + Synthetic"].sort_values("AUC", ascending=False).iloc[0]

print(f"Best (Real Only):      {best_real['Classifier']} — AUC={best_real['AUC']:.4f}, Acc={best_real['Accuracy']:.4f}")
print(f"Best (Augmented):      {best_aug['Classifier']} — AUC={best_aug['AUC']:.4f}, Acc={best_aug['Accuracy']:.4f}")
print(f"AUC Improvement:       {best_aug['AUC'] - best_real['AUC']:+.4f}")
print("="*60)